# 🚢 Port Shipping Data Cleaning Pipeline
> A complete data cleaning and transformation pipeline for maritime shipping data.  
> Raw multi-sheet Excel workbooks → clean, structured datasets ready for publishing.

## 1. 📦 Import Libraries

In [67]:
import re
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from datetime import datetime, timedelta

## 2. 📥 Load Data

We load three Excel files:
- **Countries IDs** — maps country names to their IDs
- **Port workbook** — raw shipping data across multiple country sheets
- **Ports reference** — port names, city names, codes, and IDs

In [68]:
CID     = pd.read_excel('input\\My port\\My port\\Countries IDs.xlsx')
port    = pd.read_excel('input\\My port\\My port\\MyPoert AUG sheet1.xlsx', sheet_name=None)
portand = pd.read_excel('input\\My port\\My port\\Ports_ceties_codes_and_IDs.xlsx')

## 3. 🔍 Explore Datasets

In [69]:
CID.head()

,Country,ID
0,Reunion,23
1,Fiji,28
2,Guyana,29
3,Bhutan,63
4,Solomon Islands,20


In [70]:
portand.head()

,City,Port Name,Code,ID
0,Guttenberg,A Coruña,OGX,870
1,Union City,Aalborg,FFO,229
2,West New York,Algeciras,BEH,765
3,Hoboken,Alicante,ZKN,843
4,Kaser,Almería,PWL,375


## 4. 🔗 Merge All Country Sheets

The port workbook has multiple sheets (one per country).  
We merge them all into one unified DataFrame with three columns: `trip`, `price offer`, `country`.

In [71]:
# Initialize empty DataFrame to collect all sheets
df_con = pd.DataFrame({'trip': [], 'price offer': [], 'country': []})

# Loop over each sheet and append to the main DataFrame
for sheet_name, df in port.items():
    df = df.iloc[:, :2]                      # Keep only first two columns
    df.columns = ['trip', 'price offer']     # Rename columns
    df['country'] = sheet_name               # Add country name from sheet name
    df_con = pd.concat([df_con, df])         # Append to main DataFrame

df_con

,trip,price offer,country
0,FROM Frederikshavn TO Liepaja,Trip: ONE WAY\nPrice starts from:EUR 3306\n,Reunion
1,from Emden TO Fos-sur-Mer,Trip:ONE WAY\nprice start from :EUR 6363,Reunion
2,NaN,NaN,Reunion
3,FROM Zeebrugge TO Bensersiel,Trip:ONE WAY\nprice start from :EUR 3666,Reunion
4,NaN,NaN,Reunion
...,...,...,...
29,FROM Oostende TO Norderney,Trip:ONE WAY\nprice start from :RSD 3063,Turks and Caicos Islands
30,NaN,NaN,Turks and Caicos Islands
31,FROM Oostende TO Puttgarden,Trip:ONE WAY\nprice start from :RSD 61333,Turks and Caicos Islands
32,NaN,NaN,Turks and Caicos Islands


## 5. 🧹 Clean Missing Data

### Step 1 — Drop rows where both `trip` and `price offer` are null

In [72]:
df_con.dropna(subset=['trip', 'price offer'], how='all', inplace=True)
df_con.isna().sum()

trip           14
price offer     0
country         0
dtype: int64

### Step 2 — Fix split price offer cells

Some `price offer` values are split across two rows where the second row has no `trip`.  
We merge the second row's value into the previous row, then drop the empty trip rows.

In [73]:
df_con = df_con.reset_index(drop=True)

def fix_null_trips(df):
    """
    Merges price offer text from null-trip rows into the previous row,
    then removes all rows where trip is null.
    """
    for i in range(1, len(df)):
        if pd.isna(df.loc[i, 'trip']):
            # Append current price offer to the previous row's price offer
            df.loc[i-1, 'price offer'] = str(df.loc[i-1, 'price offer']) + ' ' + str(df.loc[i, 'price offer'])
    
    # Drop rows with null trip and reset index
    df = df[df['trip'].notna()].reset_index(drop=True)
    return df

df_con = fix_null_trips(df_con)
df_con.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   trip         180 non-null    object
 1   price offer  180 non-null    object
 2   country      180 non-null    object
dtypes: object(3)
memory usage: 4.3+ KB


## 6. ✂️ Extract Features from Trip & Price Columns

We extract four new columns using string parsing and regex:
- `from` — origin port (text before 'TO')
- `to` — destination port (text after 'TO')
- `price` — numeric price extracted from price offer string
- `type` — shipment type (first letter after the colon)

In [74]:
def extract_price(x):
    """Extract the first number found in the price offer string."""
    try:
        return re.findall(r'\d+', x)[0]
    except:
        return 'not found'

def extract_from(x):
    """Extract origin port — text between position 5 and 'TO'."""
    y = x.upper().strip()[5:]
    return y[:y.index('TO')]

def extract_to(x):
    """Extract destination port — text after 'TO'."""
    y = x.upper().strip()
    return y[y.index('TO') + 3:]

def extract_type(x):
    """Extract shipment type — first character after the colon."""
    return (x[x.index(':') + 1 : x.index(':') + 4].strip())[0]

# Apply all extraction functions
df_con['from']  = df_con['trip'].apply(extract_from)
df_con['to']    = df_con['trip'].apply(extract_to)
df_con['price'] = df_con['price offer'].apply(extract_price)
df_con['type']  = df_con['price offer'].apply(extract_type)

# Check for any rows where price was not found
df_con[df_con['price'] == 'not found']

,trip,price offer,country,from,to,price,type


In [75]:
# Keep only the relevant columns
df_con = df_con[['from', 'to', 'price', 'type', 'country']]
df_con

,from,to,price,type,country
0,FREDERIKSHAVN,LIEPAJA,3306,O,Reunion
1,EMDEN,FOS-SUR-MER,6363,O,Reunion
2,ZEEBRUGGE,BENSERSIEL,3666,O,Reunion
3,EMDAN,SAGUNTO,6166,O,Reunion
4,EMDAN,MILAZZO,3161,O,Reunion
...,...,...,...,...,...
175,OOSTENDE,FREDERIKSHAVN,66636,O,Turks and Caicos Islands
176,OOSTENDE,ZEEBRUGGE,66636,O,Turks and Caicos Islands
177,OOSTENDE,NORDERNEY,3063,O,Turks and Caicos Islands
178,OOSTENDE,PUTTGARDEN,61333,O,Turks and Caicos Islands


## 7. 🌍 Fix Country Names

Some country names in our data don't match the reference Countries IDs file.  
We detect mismatches and correct them manually.

In [76]:
# Find countries not matching the reference
wrong_C = [i for i in set(df_con['country']) if i not in set(CID['Country'])]
print('Mismatched countries:', wrong_C)

Mismatched countries: ['Paramaribo', 'Majuro']


In [77]:
# Manually correct mismatched country names
df_con['country'].replace(wrong_C[0], 'Suriname',         inplace=True)
df_con['country'].replace(wrong_C[1], 'Marshall Islands', inplace=True)

In [78]:
# Merge with Countries IDs reference to get country ID
df_con_N = pd.merge(df_con, CID, left_on='country', right_on='Country', how='left')

# Verify merge — should return empty DataFrame if all matched
df_con_N[df_con_N['Country'] != df_con_N['country']]


,from,to,price,type,country,Country,ID


In [79]:
df_con_N.drop(columns=['Country'], inplace=True)
df_con_N.rename(columns={'ID': 'country id'}, inplace=True)
df_con_N.head()

,from,to,price,type,country,country id
0,FREDERIKSHAVN,LIEPAJA,3306,O,Reunion,23
1,EMDEN,FOS-SUR-MER,6363,O,Reunion,23
2,ZEEBRUGGE,BENSERSIEL,3666,O,Reunion,23
3,EMDAN,SAGUNTO,6166,O,Reunion,23
4,EMDAN,MILAZZO,3161,O,Reunion,23


## 8. 🔍 Fix Port Names with Fuzzy Matching

Some port names in our data don't exactly match the reference port list.  
We use **fuzzy string matching** (`fuzzywuzzy`) to find and fix them automatically.

In [80]:
# Strip whitespace from port columns before matching
df_con_N['from'] = df_con_N['from'].str.strip()
df_con_N['to']   = df_con_N['to'].str.strip()

def find_incorrect(df_con_N, portand):
    """
    Returns a list of port names in 'from'/'to' columns
    that don't match any port name or city in the reference table.
    """
    port_names = set(portand['Port Name'].str.strip().str.upper())
    city_names = set(portand['City'].str.strip().str.upper())
    valid = port_names | city_names

    wrong_P = set()
    for i in set(df_con_N['from'].dropna().str.strip().str.upper()):
        if i not in valid:
            wrong_P.add(i)
    for i in set(df_con_N['to'].dropna().str.strip().str.upper()):
        if i not in valid:
            wrong_P.add(i)

    return list(wrong_P)


def fix_incorrect(df_con_N, portand):
    """
    Uses fuzzy matching to find the closest port name for each wrong port,
    replaces it in the DataFrame if similarity > 60%, and reports results.
    """
    found   = []
    wrong_P = find_incorrect(df_con_N, portand)

    for p in wrong_P:
        best_match = ''
        best_sim   = 0

        for port in portand['Port Name'].str.strip():
            sim = fuzz.ratio(p.upper(), port.upper())
            if sim > best_sim and sim > 60:
                best_match = port
                best_sim   = sim

        if best_match:
            df_con_N['from'] = df_con_N['from'].str.strip().replace(p, best_match)
            df_con_N['to']   = df_con_N['to'].str.strip().replace(p, best_match)
            found.append(p)
            print(f"✅ '{p}' → '{best_match}' (similarity: {best_sim}%)")
        else:
            print(f"❌ No match found for: '{p}'")

    print(f"\nFixed : {len(found)}/{len(wrong_P)} ports")
    still_wrong = [i for i in wrong_P if i not in found]
    print(f"Still wrong: {still_wrong}")
    return df_con_N


df_con_N = fix_incorrect(df_con_N, portand)

# Verify — should return empty list if all ports matched
find_incorrect(df_con_N, portand)

✅ 'SJAELLANDS ODDE FERRY PORT' → 'Sjællands Odde Ferry Port' (similarity: 94%)
✅ 'ANTWRPEN' → 'Antwerpen' (similarity: 94%)
✅ 'FRIDERIKSHAVN' → 'Frederikshavn' (similarity: 92%)
✅ 'LIEPA'JA' → 'Liepāja' (similarity: 80%)
✅ 'LIEPAJA' → 'Liepāja' (similarity: 86%)
✅ 'ANGEOOG' → 'Langeoog' (similarity: 93%)
✅ 'ZEEBRUGG' → 'Zeebrugge' (similarity: 94%)
✅ 'FREDERYKSHAVN' → 'Frederikshavn' (similarity: 92%)
✅ 'EMDAN' → 'Emden' (similarity: 80%)
✅ 'HELSINGØR'' → 'Helsingør' (similarity: 95%)
✅ 'FREDERIKSHEVN' → 'Frederikshavn' (similarity: 92%)

Fixed : 11/11 ports
Still wrong: []


[]

## 9. 🏙️ Replace City Names with Port Names

Some entries use city names instead of port names.  
We build a city → port mapping from the reference table and apply it.

In [81]:
def replace_city_with_port(df_con_N, portand):
    """
    Replaces any city name in 'from'/'to' columns with its
    corresponding port name from the reference table.
    """
    city_to_port = dict(zip(
        portand['City'].str.strip().str.upper(),
        portand['Port Name'].str.strip()
    ))

    df_con_N['from'] = df_con_N['from'].apply(
        lambda x: city_to_port.get(x.strip().upper(), x) if pd.notna(x) else x
    )
    df_con_N['to'] = df_con_N['to'].apply(
        lambda x: city_to_port.get(x.strip().upper(), x) if pd.notna(x) else x
    )
    return df_con_N

df_con_N = replace_city_with_port(df_con_N, portand)

## 10. 🔁 Enrich with Port IDs and Codes

We merge the port reference table twice:
1. Once for the **origin** port (`from`)
2. Once for the **destination** port (`to`)

This gives us `from ID`, `from Code`, `to ID`, `to Code`.

In [ ]:
# Capitalize port names for consistent matching
df_con_N['from']         = df_con_N['from'].str.capitalize()
df_con_N['to']           = df_con_N['to'].str.capitalize()
portand['Port Name']     = portand['Port Name'].str.capitalize()

# Merge for origin port
df_con_N = pd.merge(df_con_N, portand, left_on='from', right_on='Port Name', how='left')
df_con_N.rename(columns={'ID': 'from ID', 'Code': 'from Code'}, inplace=True)
df_con_N.info()
# Merge for destination port
df_con_N = pd.merge(df_con_N, portand, left_on='to', right_on='Port Name', how='left')
df_con_N.rename(columns={'ID': 'to ID', 'Code': 'to Code'}, inplace=True)



# Drop unnecessary columns
df_con_N.drop(columns=['country', 'City_x', 'City_y'], inplace=True)

df_con_N.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 180 entries, 0 to 179
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   from         180 non-null    object
 1   to           180 non-null    object
 2   price        180 non-null    object
 3   type         180 non-null    object
 4   country id   180 non-null    int64 
 5   Port Name_x  180 non-null    object
 6   from Code    180 non-null    object
 7   from ID      180 non-null    int64 
 8   Port Name_y  180 non-null    object
 9   to Code      180 non-null    object
 10  to ID        180 non-null    int64 
dtypes: int64(3), object(8)
memory usage: 16.9+ KB


In [84]:
df_con_N.head()

,from,to,price,type,country id,Port Name_x,from Code,from ID,Port Name_y,to Code,to ID
0,Frederikshavn,Liepāja,3306,O,23,Frederikshavn,CAC,372,Liepāja,ZZK,32
1,Emden,Fos-sur-mer,6363,O,23,Emden,REV,572,Fos-sur-mer,KKA,248
2,Zeebrugge,Bensersiel,3666,O,23,Zeebrugge,YMJ,903,Bensersiel,TBY,696
3,Emden,Sagunto,6166,O,23,Emden,REV,572,Sagunto,BWM,811
4,Emden,Milazzo,3161,O,23,Emden,REV,572,Milazzo,JVZ,913


## 11. 🔗 Generate Route URLs & Add Dates

Each route gets a unique URL in the format: `FROMCODE-TOCODE-TYPE`  
We also add a publish date (today) and expiry date (37 days from today).

In [85]:
# Build port content DataFrame
port_con = df_con_N[['from ID', 'from Code', 'type', 'to ID', 'to Code']].copy()

# Generate unique route URL
port_con['url'] = port_con['from Code'] + '-' + port_con['to Code'] + '-' + port_con['type']

# Add publish and expiry dates
today = datetime.today().date()
port_con['puplish_date'] = today
port_con['expir_date']   = datetime.today() + timedelta(days=37)

# Drop code columns (no longer needed)
port_con.drop(columns=['from Code', 'to Code'], inplace=True)

# Remove duplicate routes
port_con.drop_duplicates('url', inplace=True)

port_con.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 52 entries, 0 to 165
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   from ID       52 non-null     int64         
 1   type          52 non-null     object        
 2   to ID         52 non-null     int64         
 3   url           52 non-null     object        
 4   puplish_date  52 non-null     object        
 5   expir_date    52 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 2.8+ KB


## 12. 💰 Format Prices in Multiple Languages

We format the price into a JSON string supporting 7 languages.  
Arabic prices use **Arabic-Indic numerals** (٠١٢٣٤٥٦٧٨٩).

In [86]:
def to_arabic_indic(n):
    """Convert Western numerals to Arabic-Indic numerals used in Arab countries."""
    arabic_indic = str.maketrans('0123456789', '٠١٢٣٤٥٦٧٨٩')
    return str(int(n)).translate(arabic_indic)

# Build pricing template DataFrame
port_temp = df_con_N[['from Code', 'type', 'to Code', 'country id', 'price']].copy()

# Generate URL
port_temp['url'] = port_temp['from Code'] + '-' + port_temp['to Code'] + '-' + port_temp['type']

# Format price as multi-language JSON string
port_temp['price'] = (
    '{"en":"'  + port_temp['price'] + '",'
    '"ar":'     + port_temp['price'].apply(to_arabic_indic) + ','
    '"gr":'     + port_temp['price'] + ','
    '"it":'     + port_temp['price'] + ','
    '"cz":'     + port_temp['price'] + ','
    '"fr":'     + port_temp['price'] + ','
    '"sk":'     + port_temp['price'] + '}'
)

# Keep only final columns
port_temp = port_temp[['url', 'country id', 'price']]
port_temp.head()

,url,country id,price
0,CAC-ZZK-O,23,"{""en"":""3306"",""ar"":٣٣٠٦,""gr"":3306,""it"":3306,""cz..."
1,REV-KKA-O,23,"{""en"":""6363"",""ar"":٦٣٦٣,""gr"":6363,""it"":6363,""cz..."
2,YMJ-TBY-O,23,"{""en"":""3666"",""ar"":٣٦٦٦,""gr"":3666,""it"":3666,""cz..."
3,REV-BWM-O,23,"{""en"":""6166"",""ar"":٦١٦٦,""gr"":6166,""it"":6166,""cz..."
4,REV-JVZ-O,23,"{""en"":""3161"",""ar"":٣١٦١,""gr"":3161,""it"":3161,""cz..."


## 13. 📤 Export Final Output Files

We export two clean Excel files:
- **`port_content.xlsx`** — route content with IDs, URL, and dates
- **`Port_templet.xlsx`** — pricing template with multi-language prices

In [88]:
# Export route content
port_con.to_excel('output\\port_content.xlsx', index=False)
print('✅ port_content.xlsx saved successfully')

# Export pricing template
port_temp.to_excel('output\\Port_templet.xlsx', index=False)
print('✅ Port_templet.xlsx saved successfully')

✅ port_content.xlsx saved successfully
✅ Port_templet.xlsx saved successfully
